In [1]:
%%capture

import warnings
warnings.filterwarnings("ignore")

import altair as alt
import branca.colormap as cm
import folium
import gcsfs
import pandas as pd

import gt_extras as gte

from great_tables import GT, md

import chart_utils_for_operators as chart_utils
import prep_operator_data
import gtfs_curator_utils.magics
from gtfs_curator_utils import _color_palette

from shared_vars import PREDICTIONS_GCS

alt.data_transformers.enable("vegafusion")

one_month = pd.to_datetime("2026-02-01")
date_format = "%b %Y" # gtfs_digest/_new_operator_report_utils.py
one_month_formatted = one_month.strftime(date_format)

In [2]:
operator_cols = ["tu_name", "day_type"]

schedule_cols = [
    "daily_trips", "daily_service_hours", "n_routes", "n_shapes",
    "n_stops", "num_stop_times", "daily_arrivals", 'n_days_schedule_and_rt'
]

tu_cols = ['tu_messages_per_minute', 'n_tu_trips', 'pct_tu_trips', 'pct_tu_routes'] #'daily_tu_trips',

# split prediction cols into 2 separate tables so comparison is clearer
tu_prediction_cols1 = ["bus_catch_likelihood", "pct_tu_complete_minutes", "p50", "n_predictions"]
tu_prediction_cols2 = [
    "p25", "p75", "iqr",
    "prediction_padding_minutes", "avg_prediction_spread_minutes"
]

In [3]:
equans = "Marin EQUANS Trip Update"
swiftly = "Marin Swiftly Trip Update"
mtc = "Bay Area 511 Marin TripUpdates"

df = pd.read_parquet(
    f"{PREDICTIONS_GCS}monthly_operator_summary_marin.parquet",
    filesystem=gcsfs.GCSFileSystem(),
    filters = [[("tu_name", "in", [equans, swiftly, mtc])]],
).pipe(prep_operator_data.merge_in_operator_percentiles)

In [4]:
# Set variables for color bars used across maps, route dropdown, and great tables
PREDICTION_ERROR_COLORS =list(_color_palette.PREDICTION_ERROR_COLOR_PALETTE.values())
PREDICTION_ERROR_INDEX = [-5, -3, -1, 1, 3, 5]
PREDICTION_ERROR_LEGEND_CAPTION = "minutes (negative = late; positive = early)"

POS_BAR_COLOR = _color_palette.get_color("blueberry")
NEG_BAR_COLOR = _color_palette.get_color("vivid_cerise")

## Schedule + RT Summary Stats

**Takeaways**
**1. GTFS schedule metrics are the same**

The Bay Area 511 feed is provided as a baseline for expected values for schedule data. 
Swiftly and Equans share the same schedule feed (Optibus), and we captured 23 days of RT data from Equans, but 28 days from Swiftly and Bay Area 511. 

Note: The Optibus feed might be ultimately what feeds into the Bay Area 511 feed. In our warehouse, the GMV Synchromatics has a separate name, been versioned as historic, and is not present in Feb 2026. It does seem like the Bay Area 511 and Swiftly RT feeds are the same. 

High-level schedule metrics (# trips, stops, routes, shapes, etc) are the same, with differences in total scheduled arrivals (stop_times) coming from fewer days captured with Equans.

In [5]:
schedule_table = (
    GT(df[["schedule_name"] + operator_cols + schedule_cols])
    .cols_label(
        schedule_name = "Schedule Feed",
        daily_trips = "Daily Trips",
        daily_service_hours = "Daily Service Hours",
        n_routes = "# Routes",
        n_shapes = "# Shapes",
        n_stops = "# Stops",
        num_stop_times = "Total Scheduled Arrivals",
        daily_arrivals = "Daily Scheduled Arrivals",    
        n_days_schedule_and_rt = "# days with both RT",
    ).fmt_integer(
        columns = [
            "daily_trips", "n_routes", "n_shapes", "n_stops", 
            "num_stop_times", "daily_arrivals", "n_days_schedule_and_rt"]
    ).fmt_number(
        columns = ["daily_service_hours"], decimals=1
    ).tab_spanner(
        label="Schedule",
        columns = schedule_cols
    ).tab_header(
        title = "Schedule + RT Summary Metrics", 
        subtitle = f"{one_month_formatted}"
    )
)

chart_utils.format_great_table(schedule_table, day_type_grouping = False).tab_stub(
    groupname_col="tu_name", rowname_col = "day_type")

GT(_tbl_data=                 schedule_name                         tu_name  day_type  \
0  Bay Area 511 Marin Schedule  Bay Area 511 Marin TripUpdates   Weekday   
1  Bay Area 511 Marin Schedule  Bay Area 511 Marin TripUpdates  Saturday   
2  Bay Area 511 Marin Schedule  Bay Area 511 Marin TripUpdates    Sunday   
3       Marin Optibus Schedule        Marin EQUANS Trip Update   Weekday   
4       Marin Optibus Schedule        Marin EQUANS Trip Update  Saturday   
5       Marin Optibus Schedule        Marin EQUANS Trip Update    Sunday   
6       Marin Optibus Schedule       Marin Swiftly Trip Update   Weekday   
7       Marin Optibus Schedule       Marin Swiftly Trip Update  Saturday   
8       Marin Optibus Schedule       Marin Swiftly Trip Update    Sunday   

   daily_trips  daily_service_hours  n_routes  n_shapes  n_stops  \
0        630.1                449.4        19        58      550   
1        472.0                319.7        14        37      478   
2        470.0                318.1        14        37      478   
3        627.4                447.3        19        58      550   
4        472.0                319.7        14        37      478   
5        470.0                318.1        14        37      478   
6        630.1                449.4        19        58      550   
7        472.0                319.7        14        37      478   
8        470.0                318.1        14        37      478   

   num_stop_times  daily_arrivals  n_days_schedule_and_rt  
0          320191         16009.5                      20  
1           44580         11145.0                       4  
2           44352         11088.0                       4  
3          254924         15932.8                      16  
4           44580         11145.0                       4  
5           33264         11088.0                       3  
6          320191         16009.5                      20  
7           44580         11145.0                       4  
8           44352         11088.0                       4  , _body=<great_tables._gt_data.Body object at 0x79b7c009c250>, _boxhead=Boxhead([ColInfo(var='schedule_name', type=<ColInfoTypeEnum.default: 1>, column_label='Schedule Feed', column_align='center', column_width=None), ColInfo(var='tu_name', type=<ColInfoTypeEnum.row_group: 3>, column_label='tu_name', column_align='center', column_width=None), ColInfo(var='day_type', type=<ColInfoTypeEnum.stub: 2>, column_label='day_type', column_align='center', column_width=None), ColInfo(var='daily_trips', type=<ColInfoTypeEnum.default: 1>, column_label='Daily Trips', column_align='center', column_width=None), ColInfo(var='daily_service_hours', type=<ColInfoTypeEnum.default: 1>, column_label='Daily Service Hours', column_align='center', column_width=None), ColInfo(var='n_routes', type=<ColInfoTypeEnum.default: 1>, column_label='# Routes', column_align='center', column_width=None), ColInfo(var='n_shapes', type=<ColInfoTypeEnum.default: 1>, column_label='# Shapes', column_align='center', column_width=None), ColInfo(var='n_stops', type=<ColInfoTypeEnum.default: 1>, column_label='# Stops', column_align='center', column_width=None), ColInfo(var='num_stop_times', type=<ColInfoTypeEnum.default: 1>, column_label='Total Scheduled Arrivals', column_align='center', column_width=None), ColInfo(var='daily_arrivals', type=<ColInfoTypeEnum.default: 1>, column_label='Daily Scheduled Arrivals', column_align='center', column_width=None), ColInfo(var='n_days_schedule_and_rt', type=<ColInfoTypeEnum.default: 1>, column_label='# days with both RT', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x79b770dfbad0>, _spanners=Spanners([SpannerInfo(spanner_id='Schedule', spanner_level=0, spanner_label='Schedule', spanner_units=None, spanner_pattern=None, vars=['daily_trips', 'daily_service_hours', 'n_routes', 'n_shapes', 'n_stops', 'num_stop_times', 'daily_arrivals', 'n_days_schedule_and_rt'], built=None)]), _headi

## General RT Metrics

<span style="color:#4477aa">**Update Availability Goal 1:** 2+ vehicle positions or trip updates messages per minute.</span>

<span style="color:#4477aa">**Update Availability Goal 2:** 100% routes are covered by RT, and 75%+ of trips have RT.</span>

**Takeaways**

**2. Swiftly and Equans both counted more RT trips than the Bay Area 511 feed. This is seen in Feb 2026 data, though Swiftly doesn't show the same pattern in our Jan 2026 report.**

*How schedule and RT are combined in our data warehouse*

Schedule data comes from a GTFS feed that is active for a given time period.

RT data is streamed in every 20 or 30 seconds, parsed from the json and unpacked as a table.

To link schedule with RT data, we create a composite key across the schedule feed's `base64_url`, `service_date`, `trip_id`, and `iteration_num (trip_start_time)` (for frequency-based trips) to create a scheduled_trip-to-RT_trip pairing.

*Swiftly and Equans both overcount RT trips in Feb 2026 only*

An innocuous reason could be there are trips that were not scheduled, but later added. This is one way where we would not be able to link schedule data. 

The Jan 2026 **[Swiftly report](https://analysis.dds.dot.ca.gov/rt_operator_metrics/name-marin-swiftly-trip-update/operator-report-name-marin-swiftly-trip-update/index.html)** does not show this pattern (the RT trip counts are slightly under schedule trip counts, showing that a couple of scheduled trips were missing RT data, and this is expected).
It is also not seen in the **[Bay Area 511 report](https://analysis.dds.dot.ca.gov/rt_operator_metrics/name-bay-area-511-marin-tripupdates/operator-report-name-bay-area-511-marin-tripupdate/index.html)**.

**3. Swiftly and Equans perform similarly across messages per minute and % trips and % routes with RT trip updates data.**

Trip updates per minute is a measure of *completeness of trip updates* over the trip duration.

Within each trip, a measure of how frequency of `StopTimeMessages` per minute is calculated for this metric, and a goal of at least 2 (every 30 seconds) is desirable.

It is not affected by whether RT trips are overcounted or undercounted against schedule trips.


In [6]:
rt_table1 = (
    GT(df[["schedule_name"] + operator_cols + ["n_trips"] + tu_cols])
    .cols_label(
        schedule_name = "Schedule",
        n_trips = "# Scheduled Trips",
        n_tu_trips = "# Trips (Trip Updates)",
        tu_messages_per_minute = "Messages per Minute",
        pct_tu_trips = "% Trips",
        pct_tu_routes = "% Routes",
    ).fmt_number(
        columns = ["tu_messages_per_minute"], decimals=1
    ).fmt_percent(
        columns=["pct_tu_trips", "pct_tu_routes"], decimals=1
    ).fmt_integer(columns = ["n_trips", "n_tu_trips"]
    ).tab_stub(rowname_col="day_type", groupname_col="tu_name")
    .cols_hide(columns=["tu_name"])
)

chart_utils.format_great_table(rt_table1, day_type_grouping = False).pipe(
    gte.gt_color_box, 
    columns=["tu_messages_per_minute"], 
    palette="Blues",
    domain=[1, 3]
).pipe(
    gte.gt_hulk_col_numeric, 
    columns=["pct_tu_trips", "pct_tu_routes"],
    palette=[_color_palette.get_color("light_goldenrod"), 
             _color_palette.get_color("pastel_peppermint")], 
    domain=[0, 1],
    na_color="black", # with opacity at 0.1, this is basically gray
    alpha=0.1
)

GT(_tbl_data=                 schedule_name                         tu_name  day_type  \
0  Bay Area 511 Marin Schedule  Bay Area 511 Marin TripUpdates   Weekday   
1  Bay Area 511 Marin Schedule  Bay Area 511 Marin TripUpdates  Saturday   
2  Bay Area 511 Marin Schedule  Bay Area 511 Marin TripUpdates    Sunday   
3       Marin Optibus Schedule        Marin EQUANS Trip Update   Weekday   
4       Marin Optibus Schedule        Marin EQUANS Trip Update  Saturday   
5       Marin Optibus Schedule        Marin EQUANS Trip Update    Sunday   
6       Marin Optibus Schedule       Marin Swiftly Trip Update   Weekday   
7       Marin Optibus Schedule       Marin Swiftly Trip Update  Saturday   
8       Marin Optibus Schedule       Marin Swiftly Trip Update    Sunday   

   n_trips  tu_messages_per_minute  n_tu_trips  pct_tu_trips  pct_tu_routes  
0    12602                     3.0       12530         0.994            1.0  
1     1888                     3.0        1880         0.996            1.0  
2     1880                     3.0        1866         0.993            1.0  
3    10038                     2.8       15126         1.507            1.0  
4     1888                     3.0        2802         1.484            1.0  
5     1410                     2.9        2059         1.460            1.0  
6    12602                     3.0       17642         1.401            1.0  
7     1888                     3.0        2818         1.493            1.0  
8     1880                     3.0        2535         1.349            1.0  , _body=<great_tables._gt_data.Body object at 0x79b76b482c50>, _boxhead=Boxhead([ColInfo(var='schedule_name', type=<ColInfoTypeEnum.default: 1>, column_label='Schedule', column_align='center', column_width=None), ColInfo(var='tu_name', type=<ColInfoTypeEnum.hidden: 4>, column_label='tu_name', column_align='center', column_width=None), ColInfo(var='day_type', type=<ColInfoTypeEnum.stub: 2>, column_label='day_type', column_align='center', column_width=None), ColInfo(var='n_trips', type=<ColInfoTypeEnum.default: 1>, column_label='# Scheduled Trips', column_align='center', column_width=None), ColInfo(var='tu_messages_per_minute', type=<ColInfoTypeEnum.default: 1>, column_label='Messages per Minute', column_align='center', column_width=None), ColInfo(var='n_tu_trips', type=<ColInfoTypeEnum.default: 1>, column_label='# Trips (Trip Updates)', column_align='center', column_width=None), ColInfo(var='pct_tu_trips', type=<ColInfoTypeEnum.default: 1>, column_label='% Trips', column_align='center', column_width=None), ColInfo(var='pct_tu_routes', type=<ColInfoTypeEnum.default: 1>, column_label='% Routes', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x79b76b486590>, _spanners=Spanners([]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x79b76b486710>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x79b771c4f0d0>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns='pct_tu_trips', rows=[0], mask=None), grpname=None, colname='pct_tu_trips', rownum=0, colnum=None, styles=[CellStyleText(color='#000000', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None), CellStyleFill(color='#e5f5df')]), StyleInfo(locname=LocBody(columns='pct_tu_trips', rows=[1], mask=None), grpname=None, colname='pct_tu_trips', rownum=1, colnum=None, styles=[CellStyleText(color='#000000', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None), CellStyleFill(color='#e5f5e0')]), StyleInfo(locname=LocBody(columns='pct_tu_trips', rows=[2], mask=None), grpname=None, colname='pct_tu_trips', rownum=2, colnum=None, styles=[CellStyleText(color='#000000', font=None, size=None, align=None, v_align=None, style=None, weight=None, s

### Prediction Accuracy Metrics

These metrics are derived entirely from RT trip updates (no comparison with schedule data).

**Takeaways**

**3. Equans favors early predictions and very few late predictions; Swiftly favors on-time predictions and has more late predictions.**

Bus catch likelihood is the sum of early and on-time predictions. 

Prediction error gives a sense of which way the errors tend to lean (on-time predictions would be close to 0, early predictions would be positive, late predictions would be negative). 
They can be interpreted together to understand the relative proprotions of early, on-time, and late predictions. Early and on-time predictions, however, are more desirable than late predictions.

* Equans has higher bus catch likelihood (nearly 90%) with an average prediction error of 2 minutes. More predictions are in the early / on-time group, and predictions tend to be early, with riders waiting about 2 minutes. About 10% of the predictions are late.
* Swiftly has a bus catch likelihood around 70-80%, with an average prediction error of <1 minute.
On average, it tends to get its predictions into the narrower definition of "accurate" or "on-time". Late predictions make up 20-30% of the predictions. 

*Predictions are on-time based on the last observed predicted time of arrival*

We use the last observed predicted time of arrival as when the bus arrived, **not the scheduled time of arrival**. When there is traffic, the predictions will continue to adjust to account for a delayed bus arrival. 

On-time predictions use [this definition](https://analysis.dds.dot.ca.gov/rt_operator_metrics/#reliable-prediction-accuracy), where predictions 5 minutes before must fall within narrower bounds to be considered on-time, compared to predictions 30 minutes out.


In [7]:
rt_table2 = (
    GT(df[operator_cols + tu_prediction_cols1])
    .cols_label(
        pct_tu_complete_minutes = "% Minutes with 2+ Predictions",
        bus_catch_likelihood = "Bus Catch Likelihood (Early + On-time)",
        p50 = "Prediction Error", 
        n_predictions = "# Predictions",
    ).fmt_percent(columns=["bus_catch_likelihood", "pct_tu_complete_minutes"], decimals=1)
    .fmt_number(columns=["p50"], decimals=1)
    .fmt_integer(columns=["n_predictions"])
    .tab_header(
        title = f"Trip Update Prediction Accuracy Metrics", 
        subtitle = "units are in minutes")
).cols_hide(
    columns=["tu_name"]
).pipe(
    chart_utils.format_great_table
).tab_stub(
    rowname_col="day_type", groupname_col="tu_name"
)

rt_table2.pipe(
    gte.gt_hulk_col_numeric, 
    columns=["bus_catch_likelihood", "pct_tu_complete_minutes"],
    palette=[_color_palette.get_color("light_goldenrod"), 
             _color_palette.get_color("pastel_peppermint")],
    domain=[0, 1],
    alpha=0.1
).pipe(
    gte.gt_color_box, 
    columns=["p50"], 
    palette=PREDICTION_ERROR_COLORS,
    domain=[-5, 5]  
).cols_move_to_end(columns=["n_predictions"])

GT(_tbl_data=                          tu_name  day_type  bus_catch_likelihood  \
0  Bay Area 511 Marin TripUpdates   Weekday                  0.78   
1  Bay Area 511 Marin TripUpdates  Saturday                  0.72   
2  Bay Area 511 Marin TripUpdates    Sunday                  0.71   
3        Marin EQUANS Trip Update   Weekday                  0.89   
4        Marin EQUANS Trip Update  Saturday                  0.89   
5        Marin EQUANS Trip Update    Sunday                  0.89   
6       Marin Swiftly Trip Update   Weekday                  0.78   
7       Marin Swiftly Trip Update  Saturday                  0.72   
8       Marin Swiftly Trip Update    Sunday                  0.71   

   pct_tu_complete_minutes   p50  n_predictions  
0                    0.997  0.97       31741231  
1                    0.999  0.67        3958001  
2                    0.998  0.62        4366998  
3                    0.993  2.03       17010856  
4                    0.993  2.07        2418942  
5                    0.993  1.93        1994737  
6                    0.998  0.96       31753971  
7                    0.998  0.67        3959140  
8                    0.999  0.60        4366019  , _body=<great_tables._gt_data.Body object at 0x79b76b4a7650>, _boxhead=Boxhead([ColInfo(var='tu_name', type=<ColInfoTypeEnum.row_group: 3>, column_label='tu_name', column_align='center', column_width=None), ColInfo(var='day_type', type=<ColInfoTypeEnum.stub: 2>, column_label='day_type', column_align='center', column_width=None), ColInfo(var='bus_catch_likelihood', type=<ColInfoTypeEnum.default: 1>, column_label='Bus Catch Likelihood (Early + On-time)', column_align='center', column_width=None), ColInfo(var='pct_tu_complete_minutes', type=<ColInfoTypeEnum.default: 1>, column_label='% Minutes with 2+ Predictions', column_align='center', column_width=None), ColInfo(var='p50', type=<ColInfoTypeEnum.default: 1>, column_label='Prediction Error', column_align='center', column_width=None), ColInfo(var='n_predictions', type=<ColInfoTypeEnum.default: 1>, column_label='# Predictions', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x79b76b4b05d0>, _spanners=Spanners([]), _heading=Heading(title='Trip Update Prediction Accuracy Metrics', subtitle='units are in minutes', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x79b76b4afad0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x79b7c0095ad0>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns='bus_catch_likelihood', rows=[0], mask=None), grpname=None, colname='bus_catch_likelihood', rownum=0, colnum=None, styles=[CellStyleText(color='#000000', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None), CellStyleFill(color='#ebf3cd')]), StyleInfo(locname=LocBody(columns='bus_catch_likelihood', rows=[1], mask=None), grpname=None, colname='bus_catch_likelihood', rownum=1, colnum=None, styles=[CellStyleText(color='#000000', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None), CellStyleFill(color='#ecf2c8')]), StyleInfo(locname=LocBody(columns='bus_catch_likelihood', rows=[2], mask=None), grpname=None, colname='bus_catch_likelihood', rownum=2, colnum=None, styles=[CellStyleText(color='#000000', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None), CellStyleFill(color='#edf2c7')]), StyleInfo(locname=LocBody(columns='bus_catch_likelihood', rows=[3], mask=None), grpname=None, colname='bus_catch_likelihood', rownum=3, colnum=None, styles=[CellStyleText(color='#000000', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None), CellStyleFill(color='#e8f4d7')]), StyleInfo(locna

**Takeaways**

Predictions, on average, can be on-time, with high variability, which would be frustrating for riders. These metrics untangle that variability riders might face.

The 3 feeds are ranked relative to each other. 

**4. Swiftly has less variable predictions, but riders pad the prediction more, due to higher prevalence of late predictions; Equans has more variable predictions, but because those predictions tend to be early, riders don't pad predictions as much.**

* Equans predictions tend to be more variable. The variability (75th percentile - 25th percentile) is over 4 minutes.
   * A quarter of the riders will experience late to on-time predictions (25th percentile is 0.2 minutes)
   * A quarter of riders will experience 4+ minute early predictions (75th percentile is 4.3 minutes). 
   * The middle half of riders experience predictions from 4 minute early to on-time predictions (25th to 75th percentile).
* Swiftly predictions tend to be on-time with little variability.
   * A quarter of the riders will experience late to on-time predictions (25th percentile is 0 minutes)
   * A quarter of riders will experience 2.5+ minute early predictions (75th percentile is 2.5 minutes). 
   * The middle half of riders experience predictions from 2.5 minute early to on-time predictions (25th to 75th percentile).

*How we untangle variability riders experience*
These metrics provide perspective on the distribution of prediction errors. If the average error is small and the variability is small, that is indeed promising.

The column Variability is the interquartile range (IQR). The IQR is difference between the 75th percentile and 25th percentile of prediction error.

Prediction spread is a measure of the wobble in the predictions provided (higher = predictions change more = worse rider experience).

Prediction padding is the absolute value of the 5th percentile of the prediction errors (lower = riders add less time to prevent missing bus = better). 
The idea is that riders get used to predictions always being late, and adjust their behavior to account for that. Late predictions (negative prediction error values) become the *time a rider adds to make sure they don't miss the bus*.

In [8]:
rt_table3 = (
    GT(df[operator_cols + ["p50"] + tu_prediction_cols2])
    .cols_label(
        p50 = "Prediction Error", 
        avg_prediction_spread_minutes = "Prediction Spread / Wobble",
        prediction_padding_minutes = "Prediction Padding",
        iqr = "Variability"
    )
    .fmt_number(columns=["p50", "avg_prediction_spread_minutes", "prediction_padding_minutes"], decimals=1)
    .tab_header(title = f"Trip Update Prediction Accuracy Metrics", 
                subtitle = "units are in minutes")
).pipe(
    chart_utils.format_great_table
).tab_stub(
    rowname_col="day_type", groupname_col="tu_name"
)


rt_table3.pipe(
    gte.gt_color_box, 
    columns=["p50"], 
    palette=PREDICTION_ERROR_COLORS,
    domain=[-5, 5]  
).pipe(
    gte.gt_plt_dumbbell,
    col1='p25',
    col2='p75',
    label = "IQR",
    num_decimals=1,
    col1_color=_color_palette.get_color("valentino"),
    col2_color=_color_palette.get_color("lizard_green"),
    width=100, height=50, # check these each time
    font_size=8
).pipe(
    gte.gt_color_box, 
    columns=["iqr"], 
    palette="YlOrRd"
).pipe(
    gte.gt_color_box,
    columns=["prediction_padding_minutes", "avg_prediction_spread_minutes"],
    palette="BuPu"
)

GT(_tbl_data=                          tu_name  day_type   p50   p25   p75   iqr  \
0  Bay Area 511 Marin TripUpdates   Weekday  0.97  0.03  2.48  2.45   
1  Bay Area 511 Marin TripUpdates  Saturday  0.67 -0.10  1.97  2.07   
2  Bay Area 511 Marin TripUpdates    Sunday  0.62 -0.13  1.88  2.02   
3        Marin EQUANS Trip Update   Weekday  2.03  0.23  4.52  4.28   
4        Marin EQUANS Trip Update  Saturday  2.07  0.28  4.47  4.18   
5        Marin EQUANS Trip Update    Sunday  1.93  0.20  4.25  4.05   
6       Marin Swiftly Trip Update   Weekday  0.96  0.03  2.47  2.44   
7       Marin Swiftly Trip Update  Saturday  0.67 -0.10  1.95  2.05   
8       Marin Swiftly Trip Update    Sunday  0.60 -0.13  1.88  2.01   

   prediction_padding_minutes  avg_prediction_spread_minutes  
0                        1.42                            0.2  
1                        1.82                            1.8  
2                        1.98                            0.2  
3                        0.77                            0.4  
4                        0.88                            4.4  
5                        0.72                            0.4  
6                        1.40                            0.2  
7                        1.82                            1.9  
8                        1.98                            0.2  , _body=<great_tables._gt_data.Body object at 0x79b76b48e890>, _boxhead=Boxhead([ColInfo(var='tu_name', type=<ColInfoTypeEnum.row_group: 3>, column_label='tu_name', column_align='center', column_width=None), ColInfo(var='day_type', type=<ColInfoTypeEnum.stub: 2>, column_label='day_type', column_align='center', column_width=None), ColInfo(var='p50', type=<ColInfoTypeEnum.default: 1>, column_label='Prediction Error', column_align='center', column_width=None), ColInfo(var='p25', type=<ColInfoTypeEnum.default: 1>, column_label='IQR', column_align='center', column_width=None), ColInfo(var='p75', type=<ColInfoTypeEnum.hidden: 4>, column_label='p75', column_align='center', column_width=None), ColInfo(var='iqr', type=<ColInfoTypeEnum.default: 1>, column_label='Variability', column_align='center', column_width=None), ColInfo(var='prediction_padding_minutes', type=<ColInfoTypeEnum.default: 1>, column_label='Prediction Padding', column_align='center', column_width=None), ColInfo(var='avg_prediction_spread_minutes', type=<ColInfoTypeEnum.default: 1>, column_label='Prediction Spread / Wobble', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x79b76b4dc2d0>, _spanners=Spanners([]), _heading=Heading(title='Trip Update Prediction Accuracy Metrics', subtitle='units are in minutes', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x79b76b4c0510>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x79b76b4c04d0>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x79b76b4c0490>, _formats=[<great_tables._gt_data.FormatInfo object at 0x79b76b481750>, <great_tables._gt_data.FormatInfo object at 0x79b76b4dc490>, <great_tables._gt_data.FormatInfo object at 0x79b76b4dc650>, <great_tables._gt_data.FormatInfo object at 0x79b76b4dd250>, <great_tables._gt_data.FormatInfo object at 0x79b76b4dd7d0>, <great_tables._gt_data.FormatInfo object at 0x79b76b4dd950>, <great_tables._gt_data.FormatInfo object at 0x79b76b4dda50>, <great_tables._gt_data.FormatInfo object at 0x79b76b4ddb90>, <great_tables._gt_data.FormatInfo object at 0x79b76b4ddd10>, <great_tables._gt_data.FormatInfo object at 0x79b76b4dde10>, <great_tables._gt_data.FormatInfo object at 0x79b76b4dca10>, <great_tables._gt_data.FormatInfo object at 0x79b76b4ddf50>, <great_tables._gt_data.FormatInfo object at 0x79b76b4de050>, <great_tables._gt_data.FormatInfo object at 0x79b76b4de150>, <great_tables._gt_data.FormatInfo object at 0x79b76b4de250>, <great_tables._gt_data.FormatInfo object at 0x79b76b4de350>, <great_tables._gt_data.FormatInfo

## Prediction Error Percentiles

### Distribution of Prediction Errors

The 50th percentile is the typical or median rider experience, and it can show that, on average, this transit agency is roughly on-time. 
* If the 10th percentile is fairly close to the 50th percentile, it means that the transit agency is consistent and reliable in its predictions. 
* Extreme values for the 10th percentile would indicate that predictions fluctuate, or, are somewhat unreliable.
* Steeper lines indicate fairly reliable predictions for the rider.

In [9]:
decile_cols = [
    "month_first_day", "day_type",
    "schedule_name", "tu_name",
    'pos_prediction_error_sec_array', 'pos_prediction_error_sec_percentile_array', 
    'neg_prediction_error_sec_array', 'prediction_error_sec_percentile_array'
]

operator_deciles_df = pd.read_parquet(
    f"{PREDICTIONS_GCS}monthly_operator_summary_marin.parquet",
    filesystem=gcsfs.GCSFileSystem(),
    filters = [[("tu_name", "in", [equans, swiftly, mtc])]],
    columns = decile_cols
).pipe(
    prep_operator_data.operator_deciles_for_chart
)

In [10]:
def percentile_chart_for_day_type(
    operator_deciles_df: pd.DataFrame, 
    day_type: str
) -> alt.Chart:
    """
    Create a percentile chart for each day_type + 3 feeds in the legend.
    """
    subset_df = operator_deciles_df[operator_deciles_df.day_type==day_type]
    
    percentile_chart = chart_utils.fig5and6_prediction_error_plots(
        subset_df, color_col="tu_name"
    ).properties(title=day_type, height=200, width=200)

    return percentile_chart

**Takeaways**

**5. Swiftly has tighter bounds and steeper curves for percentiles for both late and early predictions.**

* Swiftly's distribution of early and late predictions are tighter and steeper, which means that most of the predictions are on-time and don't vary as much.
    * The left and right curves mirror each other. The prediction algorithm seems agnostic to both early and late predictions.
    * Lower accuracy loss (ratio of 10th to 50th percentile) for late predictions compared to Equans.  
* Equans's tails are longer, which mean the 10th percentiles are further out, both earlier for early predictions and later for late predictions.
    * Lower accuracy loss (ratio of 10th to 50th percentile) for early predictions compared to Swiftly. 

In [11]:
day_types = ["Weekday", "Saturday", "Sunday"]

chart_list = [
    percentile_chart_for_day_type(operator_deciles_df, d) 
    for d in day_types
]

alt.hconcat(*chart_list).properties(
    title={
        "text": "Prediction Error Percentiles - Feed Comparison",
        "subtitle": "use legend to select feed"
    }
)

alt.HConcatChart(...)

### Accuracy Loss 
Ratio of the 10th to 50th percentiles 

* Newmark's paper on a small sample of transit agencies suggests that the positive prediction errors typically have ratios of 4.
* Late predictions (negative prediction errors) have ratios around 3.
* Steeper lines = less accuracy loss = better
   * y-axis is percentile (moving from 10th to 50th percentile is moving from upwards on y-axis)
   * x-axis is error (smaller change along x-axis is less accuracy loss).
   * less accuracy loss = less change along x-axis, since change along y-axis is constant (10 to 50) = steeper (unintuitive to the normal interpretation of slope!)

In [12]:
operator_cols = ["tu_name", "day_type"]
percentile_chart_cols = [
    "pos_p10", "pos_p50", "pos_error_ratio", 
    "neg_p10", "neg_p50", "neg_error_ratio"
]

mini_df = df[operator_cols + percentile_chart_cols]

# convert the 10th, 50th percentile columns to minutes
seconds_cols = ["pos_p10", "pos_p50", "neg_p10", "neg_p50"]
mini_df[seconds_cols] = mini_df[seconds_cols].divide(60).round(2)

In [13]:
mini_p10_p50_table = (
    GT(mini_df)
    .cols_label(
        pos_p10 = "10th percentile ",
        neg_p10 = "10th percentile",
        pos_p50 = "50th percentile",
        neg_p50 = "50th percentile",
        pos_error_ratio = "Accuracy Loss", 
        neg_error_ratio = "Accuracy Loss",
    ).fmt_number(
        columns=["pos_error_ratio", "neg_error_ratio"], decimals=1
    ).tab_spanner(
        label="Early Predictions (Positive Prediction Error)",
        columns=["pos_p10", "pos_p50", "pos_error_ratio"]
    ).tab_spanner(
        label="Late Predictions (Negative Prediction Error)",
        columns = ["neg_p10", "neg_p50", "neg_error_ratio"]
    )
    .tab_header(
        title = "Accuracy Loss = Ratio of 10th to 50th percentile error", 
        subtitle = "units are in minutes (lower = less accuracy loss)"
    )
).pipe(
    gte.gt_color_box, 
    columns=["pos_p10", "pos_p50", "neg_p10", "neg_p50"], 
    palette=PREDICTION_ERROR_COLORS,
    domain=[-5, 5]
).pipe(
    gte.gt_color_box,
    columns=["pos_error_ratio", "neg_error_ratio"],
    palette="YlOrRd"
)

chart_utils.format_great_table(mini_p10_p50_table).pipe(
    chart_utils.format_great_table, 
    day_type_grouping=False
).tab_stub(
    rowname_col="day_type", groupname_col="tu_name"
)

GT(_tbl_data=                          tu_name  day_type  pos_p10  pos_p50  \
0  Bay Area 511 Marin TripUpdates   Weekday     5.05     1.58   
1  Bay Area 511 Marin TripUpdates  Saturday     4.37     1.37   
2  Bay Area 511 Marin TripUpdates    Sunday     4.32     1.30   
3        Marin EQUANS Trip Update   Weekday     7.71     2.97   
4        Marin EQUANS Trip Update  Saturday     7.57     2.93   
5        Marin EQUANS Trip Update    Sunday     7.33     2.83   
6       Marin Swiftly Trip Update   Weekday     5.03     1.58   
7       Marin Swiftly Trip Update  Saturday     4.35     1.36   
8       Marin Swiftly Trip Update    Sunday     4.30     1.28   

   pos_error_ratio  neg_p10  neg_p50  neg_error_ratio  
0              3.2    -2.38    -0.60              4.0  
1              3.2    -2.58    -0.65              4.0  
2              3.3    -2.80    -0.68              4.1  
3              2.6    -4.21    -0.67              6.3  
4              2.6    -4.02    -0.70              5.7  
5              2.6    -5.06    -0.66              7.6  
6              3.2    -2.37    -0.58              4.1  
7              3.2    -2.57    -0.65              3.9  
8              3.3    -2.78    -0.67              4.1  , _body=<great_tables._gt_data.Body object at 0x79b75b340910>, _boxhead=Boxhead([ColInfo(var='tu_name', type=<ColInfoTypeEnum.row_group: 3>, column_label='tu_name', column_align='center', column_width=None), ColInfo(var='day_type', type=<ColInfoTypeEnum.stub: 2>, column_label='day_type', column_align='center', column_width=None), ColInfo(var='pos_p10', type=<ColInfoTypeEnum.default: 1>, column_label='10th percentile ', column_align='center', column_width=None), ColInfo(var='pos_p50', type=<ColInfoTypeEnum.default: 1>, column_label='50th percentile', column_align='center', column_width=None), ColInfo(var='pos_error_ratio', type=<ColInfoTypeEnum.default: 1>, column_label='Accuracy Loss', column_align='center', column_width=None), ColInfo(var='neg_p10', type=<ColInfoTypeEnum.default: 1>, column_label='10th percentile', column_align='center', column_width=None), ColInfo(var='neg_p50', type=<ColInfoTypeEnum.default: 1>, column_label='50th percentile', column_align='center', column_width=None), ColInfo(var='neg_error_ratio', type=<ColInfoTypeEnum.default: 1>, column_label='Accuracy Loss', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x79b7681e6a90>, _spanners=Spanners([SpannerInfo(spanner_id='Early Predictions (Positive Prediction Error)', spanner_level=0, spanner_label='Early Predictions (Positive Prediction Error)', spanner_units=None, spanner_pattern=None, vars=['pos_p10', 'pos_p50', 'pos_error_ratio'], built=None), SpannerInfo(spanner_id='Late Predictions (Negative Prediction Error)', spanner_level=0, spanner_label='Late Predictions (Negative Prediction Error)', spanner_units=None, spanner_pattern=None, vars=['neg_p10', 'neg_p50', 'neg_error_ratio'], built=None)]), _heading=Heading(title='Accuracy Loss = Ratio of 10th to 50th percentile error', subtitle='units are in minutes (lower = less accuracy loss)', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x79b771a04590>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x79b76be2e810>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x79b75b383210>, _formats=[<great_tables._gt_data.FormatInfo object at 0x79b75b3832d0>, <great_tables._gt_data.FormatInfo object at 0x79b75b381e10>, <great_tables._gt_data.FormatInfo object at 0x79b75b381c10>, <great_tables._gt_data.FormatInfo object at 0x79b75b380e10>, <great_tables._gt_data.FormatInfo object at 0x79b75b381010>, <great_tables._gt_data.FormatInfo object at 0x79b75b380e90>, <great_tables._gt_data.FormatInfo object at 0x79b75b381190>, <great_tables._gt_data.FormatInfo object at 0x79b75b381350>, <great_tables._gt_data.FormatInfo object at 0x79b75b381390>, <great_tables._gt_data.FormatInfo obje